Generating arrays for h5 file

In [1]:
import glob
import h5py
import importlib
import IPython.display as ipd
import soxr
import numpy as np
import os
import pandas as pd
import pickle
import soundfile as sf
import src.audio_transforms as at
import src.custom_modules as cm
import src.spatial_attn_lightning as binaural_lightning
importlib.reload(binaural_lightning)
import sys
import torch
import tqdm
import yaml

from pathlib import Path
from pytorch_lightning import Trainer
from scipy import signal
from scipy.io.wavfile import read, write

sys.path.append('../')
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

In [2]:
# path to 376 wiki words
swc_path = '/om2/user/msaddler/spatial_audio_pipeline/assets/human_experiment_v00/foreground_swc/'

# path o common voice words not included in training
df = pd.read_pickle('/om2/user/imgriff/datasets/spatial_audio_pipeline/assets/commonvoice_9_en/manifest_all_word_alignments.pdpkl')

In [3]:
manifest = pd.read_pickle(swc_path + 'manifest.pdpkl')

In [2]:
fn_pkl_src = '/scratch2/weka/mcdermott/raygon/projects/public/jsinDataset/assets/data/interim/swc/mungedFinalDataframeWords_swc_readerNormalized_sexNormalized_accent.pdpkl'
fn_pkl_dst = '/om2/user/msaddler/spatial_audio_pipeline/assets/swc/manifest_all_words.pdpkl'

In [3]:
word_dict = pickle.load(open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_800_word_label_to_int_dict.pkl", 'rb'))
words = list(word_dict.keys())
words = [word.replace("'", "") for word in words]
manifest_all_words = pd.read_pickle(fn_pkl_dst)
# filter out words not in 'words' list
manifest_all_words = manifest_all_words[manifest_all_words['word'].isin(words)]
word_counts = manifest_all_words['word'].value_counts()

In [29]:
down_df = manifest_all_words.groupby(['word', 'gender']).sample(1, replace=False)

In [ ]:
final_df = down_df.groupby('word').filter(lambda x: len(x) == 2)

In [ ]:
final_df = final_df.reset_index().rename(columns={'index':'src_ix'})

In [ ]:
final_df

In [4]:
all_words_not_filtered = pd.read_pickle(fn_pkl_dst)
oov_words = all_words_not_filtered[~all_words_not_filtered['word'].isin(words)]

In [ ]:
talkers = final_df['client_id'].unique()
oov_words[oov_words['client_id'].isin(talkers)].client_id.value_counts()

In [ ]:
samples_per_talker = {talker:count for talker,count in final_df.client_id.value_counts().items()}
viables_cues = oov_words[oov_words.client_id.isin(talkers)]

cues = viables_cues.groupby('client_id').apply(lambda group: group.sample(samples_per_talker[group.name]))
cues.drop(columns='client_id', inplace=True)
cues = cues.reset_index()
cues.rename(columns={'level_1':'cue_src_ix'}, inplace=True)

In [ ]:
final_df.sort_values(by='client_id', inplace=True)
final_df.reset_index(inplace=True, drop=True)


cues.sort_values(by='client_id', inplace=True)
cues.reset_index(inplace=True, drop=True)



### Merge cues with foregrounds  
cues[['cue_word', 'cue_src_ix', 'cue_client_id', 'cue_fn', 'cue_start_in_s', 'cue_end_in_s']] = cues[['word', 'cue_src_ix', 'client_id', 'src_fn', 'clip_start_in_s', 'clip_end_in_s']]
# Combine as experiment dataframe
training_df = final_df.join(cues[['cue_word', 'cue_src_ix', 'cue_client_id', 'cue_fn', 'cue_start_in_s', 'cue_end_in_s']])
assert (training_df['client_id'] == training_df['cue_client_id']).all(), "Cue and Target talkers don't match!"

In [ ]:
training_df

In [5]:
training_df = pd.read_pickle('/om2/user/rphess/Auditory-Attention/binaural_test_manifest.pdpkl')

Trial run of Testing (if this works, will spatialize the manifest and write test script)

In [6]:
def pad_or_trim_to_len(x, n, mode='both', kwargs_pad={}):
    """
    Increases or decreases the length of a one-dimensional signal
    by either padding or triming the array. If the difference
    between `len(x)` and `n` is odd, this function will default to
    adding/removing the extra sample at the end of the signal.
    
    Args
    ----
    x (np.ndarray): one-dimensional input signal
    n (int): length of output signal
    mode (str): specify which end of signal to modify
        (default behavior is to symmetrically modify both ends)
    kwargs_pad (dict): keyword arguments for np.pad function
    
    Returns
    -------
    x_out (np.ndarray): one-dimensional signal with length `n`
    """
    assert len(np.array(x).shape) == 1, "input must be 1D array"
    assert mode.lower() in ['both', 'start', 'end']
    n_diff = np.abs(len(x) - n)
    if len(x) > n:
        if mode.lower() == 'end':
            x_out = x[:n]
        elif mode.lower() == 'start':
            x_out = x[-n:]
        else:
            x_out = x[int(np.floor(n_diff / 2)):-int(np.ceil(n_diff / 2))]
    elif len(x) < n:
        if mode.lower() == 'end':
            pad_width = [0, n_diff]
        elif mode.lower() == 'start':
            pad_width = [n_diff, 0]
        else:
            pad_width = [int(np.floor(n_diff / 2)), int(np.ceil(n_diff / 2))]
        kwargs = {'mode': 'constant'}
        kwargs.update(kwargs_pad)
        x_out = np.pad(x, pad_width, **kwargs)
    else:
        x_out = x
    assert len(x_out) == n
    return x_out

In [7]:
def get_excerpt(dfi, dur=3.0, sr=50000, pad_with_context=True, jitter_fraction=0):
    """
    This function loads an audio file and excerpts a clip with the specified
    duration. Target durations that exceed clip boundaries are handled with
    zero-padding (applied to all signals but sliced away when not needed).
    This function also handles resampling (via soxr) and re-scaling.
    """
    jitter_in_s = 0
    jitter_via_zero_padding = True
    if dfi.clip_dur_in_s > dur:
        # Take a random segment if clip duration is longer than excerpt
        clip_start_in_s = np.random.uniform(
            low=dfi.clip_start_in_s,
            high=dfi.clip_start_in_s + dfi.clip_dur_in_s - dur,
            size=None)
        clip_end_in_s = clip_start_in_s + dur
        jitter_via_zero_padding = False
    else:
        # Temporally jitter clip by extending either start or end time
        jitter_in_s = np.random.uniform(
            low=-dfi.clip_dur_in_s * jitter_fraction,
            high=dfi.clip_dur_in_s * jitter_fraction,
            size=None)
        if pad_with_context:
            # If using context, adjust clip start and end times to account for jitter and context
            if jitter_in_s > 0:
                clip_start_in_s = dfi.clip_start_in_s - (2 * np.abs(jitter_in_s))
                clip_end_in_s = dfi.clip_end_in_s
            else:
                clip_start_in_s = dfi.clip_start_in_s
                clip_end_in_s = dfi.clip_end_in_s + (2 * np.abs(jitter_in_s))
            clip_dur_in_s = clip_end_in_s - clip_start_in_s
            jitter_via_zero_padding = False
            context_pad_in_s = (dur - clip_dur_in_s) / 2
        else:
            clip_start_in_s = dfi.clip_start_in_s
            clip_end_in_s = dfi.clip_end_in_s
            context_pad_in_s = 0
        clip_start_in_s = clip_start_in_s - context_pad_in_s
        clip_end_in_s = clip_end_in_s + context_pad_in_s
    clip_dur_in_s = clip_end_in_s - clip_start_in_s
    # Load audio, pad, slice with indexes that account for padding
    load_full_file = True
    if (clip_start_in_s >= 0) and (clip_end_in_s < dfi.total_file_duration_in_s):
        # Attempt to read only the specified excerpt
        myfile = sf.SoundFile(dfi.src_fn)
        if myfile.seekable():
            src_sr = myfile.samplerate
            frame_start = int(np.round(clip_start_in_s * src_sr))
            frames = int(np.round(clip_dur_in_s * src_sr))
            myfile.seek(frame_start)
            y = myfile.read(frames, always_2d=True)
            y = np.mean(y, axis=1)
            load_full_file = False
    if load_full_file:
        # If impossible, read full audio file
        y, src_sr = sf.read(dfi.src_fn, always_2d=True)
        y = np.mean(y, axis=1)
        frame_start = int(np.round(clip_start_in_s * src_sr))
        frames = int(np.round(clip_dur_in_s * src_sr))
        if frame_start < 0:
            y = np.pad(y, [-frame_start, 0])
            frame_start = 0
        if frame_start + frames > len(y):
            y = np.pad(y, [0, frame_start + frames - len(y)])
        y = y[frame_start : frame_start + frames]
    # Resample from src_sr to sr
    y = soxr.resample(y, src_sr, sr).astype(np.float32)
    # If not yet jittered, apply jitter at end via asymmetric zero-padding
    if jitter_via_zero_padding:
        jitter_pad_width = int(np.round(2 * np.abs(jitter_in_s) * sr))
        if jitter_in_s > 0:
            y = np.pad(y, [jitter_pad_width, 0]).astype(np.float32)
        else:
            y = np.pad(y, [0, jitter_pad_width]).astype(np.float32)
    # Zero-pad or trim to length (fixes off by one errors)
    y = pad_or_trim_to_len(y, int(dur * sr))
    y = np.nan_to_num(y.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    return y

In [8]:
df = training_df.sample(800)

In [9]:
foregrounds = []
cues = []
for i, row in df.iterrows():
    src_ix = row['src_ix']
    cue_ix = row['cue_src_ix']
    src_row = all_words_not_filtered.iloc[src_ix]
    cue_row = all_words_not_filtered.iloc[cue_ix]
    foregrounds.append(get_excerpt(src_row))
    cues.append(get_excerpt(cue_row))

In [10]:
df['loaded_foreground'] = foregrounds

In [11]:
df['loaded_cue'] = cues

In [18]:
mdl_ckpt = '/net/vast-storage/scratch/vast/mcdermott/rphess/Auditory-Attention/attn_cue_models/word_task_voice_cue/checkpoints/epoch=0-step=1250-v1.ckpt'

In [13]:
config = yaml.load(open('/om2/user/rphess/Auditory-Attention/config/binaural_attn/word_task_voice_cue.yml', 'r'), Loader=yaml.FullLoader)

In [14]:
import src.spatial_attn_lightning as attn_tracking_lightning

In [15]:
config['noise_kwargs']['low_snr'] = 0
config['noise_kwargs']['high_snr'] = 0

In [19]:
model = attn_tracking_lightning.BinauralAttentionModule.load_from_checkpoint(checkpoint_path=mdl_ckpt, config=config).cuda()

num_classes={'num_words': 800}
Model performing word task
cochlea_filt {'sr': 50000, 'env_sr': 10000, 'n_channels': 40, 'low_lim': 40, 'use_pad': True, 'binaural': True, 'rep_on_gpu': True, 'center_crop': True, 'out_dur': 2, 'impulse_len': 0.25, 'env_extraction_type': 'Half-wave Rectification', 'downsampling_type': 'TorchTransformsResample', 'downsampling_kwargs': {'lowpass_filter_width': 64, 'rolloff': 0.9475937167399596, 'resampling_method': 'kaiser_window', 'beta': 14.769656459379492}} coch_p3 {'scale': 1, 'offset': 1e-07, 'clip_value': 5, 'power': 0.3}
center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram


Case 1: target at 0, distractor at 90
Case 2: target at -90, distractor at 90
Case 3: both at 0

In [17]:
def mass_spatialize(words, ir):
    """Uses pytorch to convolve all sounds in words with 2 channel IR given in ir"""
    n_words = words.shape[0]
    words_padded = [torch.nn.functional.pad(word, (ir.shape[0] - 1, 0)) for word in words]
    ir = ir.T.unsqueeze(1)
    words_padded = torch.stack(words_padded)
    spatialized = torch.nn.functional.conv1d(words_padded.view(n_words, 1, -1).cuda(), ir.cuda()).cuda()
    return spatialized

In [26]:
print("Loading speaker array room BRIRs")
list_data_dict = []
for elev in [-20, -10, 0, 10, 20, 30, 40]:
    for azim in np.arange(0, 360, 5):
        data_dict = {
            'azim': azim,
            'elev': elev,
            'brir': [],
        }
        for ear in ['l', 'r']:
            basename = f'{elev}elev_{azim}az_2.47x2.60y2.00z_{ear}.wav'
            if elev >= 0:
                fn = os.path.join('/om/user/francl/Room_Simulator_20181115_Rebuild/room_HRIRs/', basename)
            else:
                fn = os.path.join('/om/user/francl/Room_Simulator_20181115_Rebuild/room_HRIRs/neg_elevs/', basename)
            assert os.path.exists(fn)
            brir, sr_src = sf.read(fn)
            sr = 50000
            brir = soxr.resample(brir.astype(np.float32), sr_src, sr)
            data_dict['brir'].append(brir)
        data_dict['sr'] = sr
        data_dict['brir'] = np.stack(data_dict['brir']).T
        list_data_dict.append(data_dict)
df_brir = pd.DataFrame(list_data_dict)
df_brir_room = df_brir[np.logical_and.reduce([
    df_brir['azim'] % 10 == 0,
    ~(np.logical_and(df_brir['azim'] > 90, df_brir['azim'] < 270)),
    df_brir['elev'] >= 0,
])].reset_index()
print(f"Loaded speaker array room BRIRs ({len(df_brir_room)})")

Loading speaker array room BRIRs
Loaded speaker array room BRIRs (95)


In [27]:
brir_00 = torch.from_numpy(df_brir_room[(df_brir_room['azim'] == 0) & (df_brir_room['elev'] == 0)]['brir'].values[0])
brir_900 = torch.from_numpy(df_brir_room[(df_brir_room['azim'] == 270) & (df_brir_room['elev'] == 0)]['brir'].values[0])
brir_neg900 = torch.from_numpy(df_brir_room[(df_brir_room['azim'] == 90) & (df_brir_room['elev'] == 0)]['brir'].values[0])

In [28]:
brir_00 = torch.flip(brir_00, dims=[0])
brir_900 = torch.flip(brir_900, dims=[0])
brir_neg900 = torch.flip(brir_neg900, dims=[0])

In [21]:
audio_transforms = model.audio_transforms

In [22]:
class_map = pickle.load( open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_800_word_label_to_int_dict.pkl", "rb" )) 
class_map = {k.replace("'", ''):v for k,v in class_map.items()}

In [95]:
model.config['num_workers'] = 0
model.config['hparas']['batch_size'] = 8

In [60]:
dataloader = model.val_dataloader()

49 files in val concat dataset


In [61]:
# testing with the model val loader
results = []
for i, batch in tqdm.tqdm(enumerate(dataloader), total=100):
    cue, cue_mask_ix, scene, label = batch

    out = model.forward(cue.cuda(), scene.cuda(), None)
    softmax_outputs = torch.nn.functional.softmax(out, dim=-1)
    result = torch.argmax(softmax_outputs, dim=-1).cpu()
    results.append(label == result)
    if i == 100:
        break
result_tensor = torch.concat(results)
result_tensor.sum() / len(result_tensor)


100%|██████████| 100/100 [03:22<00:00,  2.02s/it]


In [58]:
rev_map = {v:k for k,v in class_map.items()}

In [64]:
rev_map[result]

'little'

In [152]:
# case 1
results = []
for i, row in tqdm.tqdm(df.iterrows(), total = 350):
    cue = torch.from_numpy(row['loaded_cue']).unsqueeze(0)
    fg = torch.from_numpy(row['loaded_foreground']).unsqueeze(0)
    client_id = row['client_id']
    word = row['word']
    label = class_map[word]
    distractor = torch.from_numpy(df[df['client_id'] != client_id]['loaded_foreground'].sample(1).values[0]).unsqueeze(0)

    cue = mass_spatialize(cue.cuda(), brir_00.cuda()).cpu()
    cue = np.array(cue[:, :, 12500:137500])
    cue = audio_transforms(cue, None)[0].squeeze(0)

    fg = mass_spatialize(fg.cuda(), brir_00.cuda()).cpu()
    fg = np.array(fg[:, :, 12500:137500])
    bg = mass_spatialize(distractor.cuda(), brir_900.cuda()).cpu()
    bg = np.array(bg[:, :, 12500:137500])
    scene = audio_transforms(fg, bg)[0].squeeze(0)

    out = model.forward(cue.cuda(), scene.cuda(), None)
    softmax_outputs = torch.nn.functional.softmax(out, dim=-1)
    result = int(torch.argmax(softmax_outputs, dim=-1).cpu())
    results.append(label == result)
np.mean(results)

100%|██████████| 350/350 [02:23<00:00,  2.44it/s]


0.16857142857142857

In [58]:
results = []
confusions = []
for i, row in tqdm.tqdm(df.iterrows(), total = 500):
    cue = torch.from_numpy(row['loaded_cue']).unsqueeze(0)
    fg = torch.from_numpy(row['loaded_foreground']).unsqueeze(0)
    client_id = row['client_id']
    word = row['word']
    gender = row['gender']
    label = class_map[word]
    distractor = df[(df['client_id'] != client_id) & (df['gender'] != gender)].sample(1)
    distractor_signal = torch.from_numpy(distractor['loaded_foreground'].values[0]).unsqueeze(0)
    distractor_label = class_map[distractor['word'].values[0]]
    # distractor = torch.from_numpy(df[df['client_id'] != client_id]['loaded_foreground'].sample(1).values[0]).unsqueeze(0)

    cue = mass_spatialize(cue.cuda(), brir_00.cuda()).cpu()
    cue = np.array(cue[:, :, 12500:137500])
    cue = audio_transforms(cue, None)[0].squeeze(0)

    fg = mass_spatialize(fg.cuda(), brir_900.cuda()).cpu()
    fg = np.array(fg[:, :, 12500:137500])
    bg = mass_spatialize(distractor_signal.cuda(), brir_neg900.cuda()).cpu()
    bg = np.array(bg[:, :, 12500:137500])
    scene = audio_transforms(fg, bg)[0].squeeze(0)

    out = model.forward(cue.cuda(), scene.cuda(), None)
    softmax_outputs = torch.nn.functional.softmax(out, dim=-1)
    result = int(torch.argmax(softmax_outputs, dim=-1).cpu())
    results.append(label == result)
    confusions.append(distractor_label == result)
print(f"Accuracy = {np.mean(results)}")
print(f"Confusions = {np.mean(confusions)}")

100%|██████████| 500/500 [03:22<00:00,  2.47it/s]

Accuracy = 0.156
Confusions = 0.054


In [59]:
print(f"Accuracy = {np.mean(results)}")
print(f"Confusions = {np.mean(confusions)}")

Accuracy = 0.156
Confusions = 0.054


In [29]:
results = []
confusions = []
for i, row in tqdm.tqdm(df.iterrows(), total = 500):
    cue = torch.from_numpy(row['loaded_cue']).unsqueeze(0)
    fg = torch.from_numpy(row['loaded_foreground']).unsqueeze(0)
    client_id = row['client_id']
    word = row['word']
    gender = row['gender']
    label = class_map[word]
    distractor = df[(df['client_id'] != client_id) & (df['gender'] == gender)].sample(1)
    distractor_signal = torch.from_numpy(distractor['loaded_foreground'].values[0]).unsqueeze(0)
    distractor_label = class_map[distractor['word'].values[0]]
    # distractor = torch.from_numpy(df[df['client_id'] != client_id]['loaded_foreground'].sample(1).values[0]).unsqueeze(0)

    cue = mass_spatialize(cue.cuda(), brir_00.cuda()).cpu()
    cue = np.array(cue[:, :, 12500:137500])
    cue = audio_transforms(cue, None)[0].squeeze(0)

    fg = mass_spatialize(fg.cuda(), brir_900.cuda()).cpu()
    fg = np.array(fg[:, :, 12500:137500])
    bg = mass_spatialize(distractor_signal.cuda(), brir_neg900.cuda()).cpu()
    bg = np.array(bg[:, :, 12500:137500])
    scene = audio_transforms(fg, None)[0].squeeze(0)

    out = model.forward(cue.cuda(), scene.cuda(), None)
    softmax_outputs = torch.nn.functional.softmax(out, dim=-1)
    result = int(torch.argmax(softmax_outputs, dim=-1).cpu())
    results.append(label == result)
    confusions.append(distractor_label == result)
print(f"Accuracy = {np.mean(results)}")
print(f"Confusions = {np.mean(confusions)}")

800it [02:30,  5.33it/s]                         

Accuracy = 0.56625
Confusions = 0.0


In [30]:
print(f"Accuracy = {np.mean(results)}")
print(f"Confusions = {np.mean(confusions)}")

Accuracy = 0.56625
Confusions = 0.0


In [49]:
results = []
confusions = []
for i, row in tqdm.tqdm(df.iterrows(), total = 350):
    
    cue = np.repeat(row['loaded_cue'][np.newaxis,:], 2, axis=0).reshape(1,2,-1)
    fg = np.repeat(row['loaded_foreground'][np.newaxis,:], 2, axis=0).reshape(1,2,-1)
    client_id = row['client_id']
    gender = row['gender']
    word = row['word']
    label = class_map[word]
    distractor = df[(df['client_id'] != client_id) & (df['gender'] != gender)].sample(1)
    bg = np.repeat(distractor['loaded_foreground'].item()[np.newaxis,:], 2, axis=0).reshape(1,2,-1)
    distractor_label = class_map[distractor['word'].values[0]]
    # distractor = torch.from_numpy(df[df['client_id'] != client_id]['loaded_foreground'].sample(1).values[0]).unsqueeze(0)

    # cue = mass_spatialize(cue.cuda(), brir_00.cuda()).cpu()
    cue = np.array(cue[:, :, 12500:137500])
    cue = audio_transforms(cue, None)[0].squeeze(0)

    # fg = mass_spatialize(fg.cuda(), brir_00.cuda()).cpu()
    fg = np.array(fg[:, :, 12500:137500])
    # bg = mass_spatialize(distractor_signal.cuda(), brir_900.cuda()).cpu()
    bg = np.array(bg[:, :, 12500:137500])
    scene = audio_transforms(fg, bg)[0].squeeze(0)

    out = model.forward(cue.cuda(), scene.cuda(), None)
    softmax_outputs = torch.nn.functional.softmax(out, dim=-1)
    result = int(torch.argmax(softmax_outputs, dim=-1).cpu())
    results.append(label == result)
    confusions.append(distractor_label == result)
print(f"Accuracy = {np.mean(results)}")
print(f"Confusions = {np.mean(confusions)}")

500it [02:19,  3.57it/s]                         

Accuracy = 0.296
Confusions = 0.052


In [50]:
print(f"Accuracy = {np.mean(results)}")
print(f"Confusions = {np.mean(confusions)}")

Accuracy = 0.296
Confusions = 0.052


In [51]:
results = []
confusions = []
for i, row in tqdm.tqdm(df.iterrows(), total = 501):
    
    cue = np.repeat(row['loaded_cue'][np.newaxis,:], 2, axis=0).reshape(1,2,-1)
    fg = np.repeat(row['loaded_foreground'][np.newaxis,:], 2, axis=0).reshape(1,2,-1)
    client_id = row['client_id']
    gender = row['gender']
    word = row['word']
    label = class_map[word]
    distractor = df[(df['client_id'] != client_id) & (df['gender'] == gender)].sample(1)
    bg = np.repeat(distractor['loaded_foreground'].item()[np.newaxis,:], 2, axis=0).reshape(1,2,-1)
    distractor_label = class_map[distractor['word'].values[0]]
    # distractor = torch.from_numpy(df[df['client_id'] != client_id]['loaded_foreground'].sample(1).values[0]).unsqueeze(0)

    # cue = mass_spatialize(cue.cuda(), brir_00.cuda()).cpu()
    cue = np.array(cue[:, :, 12500:137500])
    cue = audio_transforms(cue, None)[0].squeeze(0)

    # fg = mass_spatialize(fg.cuda(), brir_00.cuda()).cpu()
    fg = np.array(fg[:, :, 12500:137500])
    # bg = mass_spatialize(distractor_signal.cuda(), brir_900.cuda()).cpu()
    bg = np.array(bg[:, :, 12500:137500])
    scene = audio_transforms(fg, bg)[0].squeeze(0)

    out = model.forward(cue.cuda(), scene.cuda(), None)
    softmax_outputs = torch.nn.functional.softmax(out, dim=-1)
    result = int(torch.argmax(softmax_outputs, dim=-1).cpu())
    results.append(label == result)
    confusions.append(distractor_label == result)
print(f"Accuracy = {np.mean(results)}")
print(f"Confusions = {np.mean(confusions)}")

100%|█████████▉| 500/501 [02:20<00:00,  3.56it/s]

Accuracy = 0.198
Confusions = 0.106


In [23]:
results = []
confusions = []
for i, row in tqdm.tqdm(df.iterrows(), total = 501):
    
    cue = np.repeat(row['loaded_cue'][np.newaxis,:], 2, axis=0).reshape(1,2,-1)
    fg = np.repeat(row['loaded_foreground'][np.newaxis,:], 2, axis=0).reshape(1,2,-1)
    client_id = row['client_id']
    gender = row['gender']
    word = row['word']
    label = class_map[word]
    distractor = df[(df['client_id'] != client_id) & (df['gender'] == gender)].sample(1)
    bg = np.repeat(distractor['loaded_foreground'].item()[np.newaxis,:], 2, axis=0).reshape(1,2,-1)
    distractor_label = class_map[distractor['word'].values[0]]
    # distractor = torch.from_numpy(df[df['client_id'] != client_id]['loaded_foreground'].sample(1).values[0]).unsqueeze(0)

    # cue = mass_spatialize(cue.cuda(), brir_00.cuda()).cpu()
    cue = np.array(cue[:, :, 12500:137500])
    cue = audio_transforms(cue, None)[0].squeeze(0)

    # fg = mass_spatialize(fg.cuda(), brir_00.cuda()).cpu()
    fg = np.array(fg[:, :, 12500:137500])
    # bg = mass_spatialize(distractor_signal.cuda(), brir_900.cuda()).cpu()
    bg = np.array(bg[:, :, 12500:137500])
    scene = audio_transforms(fg, None)[0].squeeze(0)

    out = model.forward(cue.cuda(), scene.cuda(), None)
    softmax_outputs = torch.nn.functional.softmax(out, dim=-1)
    result = int(torch.argmax(softmax_outputs, dim=-1).cpu())
    results.append(label == result)
    confusions.append(distractor_label == result)
print(f"Accuracy = {np.mean(results)}")
print(f"Confusions = {np.mean(confusions)}")

800it [01:39,  8.02it/s]                         

Accuracy = 0.7025
Confusions = 0.0


In [24]:
print(f"Accuracy = {np.mean(results)}")
print(f"Confusions = {np.mean(confusions)}")

Accuracy = 0.7025
Confusions = 0.0
